# FlashAttention Demo (RTX 3050)

This notebook demonstrates **FlashAttention**: computing
**attention(Q, K, V) = softmax(QKᵀ/√d) V**
without materializing the full S×S attention matrix in global memory.

- **Tiling**: Q in BLOCK_M rows, K/V in BLOCK_N rows; blocks loaded into SRAM.
- **Online softmax**: Running max and sum so we never store full rows of scores.
- **Memory**: O(S) instead of O(S²).

**Run from repo root.** Uses Triton implementation; optional CUDA in `flash_attention_simple/` (requires JIT build).

In [ ]:
import sys
import time
from pathlib import Path
root = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## Algorithm: FlashAttention tiling

1. Partition Q into blocks of **BLOCK_M** rows, K and V into blocks of **BLOCK_N** rows.
2. For each Q block: initialize running max m_i and sum l_i (online softmax), accumulator acc.
3. For each K/V block: load Q_tile, K_tile, V_tile; compute scores = Q_tile @ K_tileᵀ × scale; update m_i, l_i, acc using online softmax; acc += softmax(scores) @ V_tile.
4. Write acc / l_i as output block. **No full S×S matrix** is ever in global memory.

## Example: Triton FlashAttention usage and correctness

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA required")

from triton_kernels.flash_attention import flash_attention_triton

B, H, S, D = 2, 4, 128, 64
q = torch.randn(B, H, S, D, device="cuda", dtype=torch.float16)
k = torch.randn(B, H, S, D, device="cuda", dtype=torch.float16)
v = torch.randn(B, H, S, D, device="cuda", dtype=torch.float16)

out_triton = flash_attention_triton(q, k, v, causal=False)
scale = 1.0 / (D ** 0.5)
out_ref = torch.nn.functional.scaled_dot_product_attention(
    q.float(), k.float(), v.float(), scale=scale, is_causal=False
).half()
err = (out_triton - out_ref).abs().max().item()
print(f"FlashAttention Triton shape: {out_triton.shape}, max diff vs SDPA: {err}")

## Benchmark: PyTorch SDPA vs Triton FlashAttention

In [ ]:
def run_bench(fn, warmup=5, repeat=25):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeat):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000 / repeat

configs = [(2, 8, 128, 64), (4, 8, 256, 64), (2, 8, 512, 64)]
results = []
for B, H, S, D in configs:
    q = torch.randn(B, H, S, D, device="cuda", dtype=torch.float16)
    k = torch.randn(B, H, S, D, device="cuda", dtype=torch.float16)
    v = torch.randn(B, H, S, D, device="cuda", dtype=torch.float16)
    t_sdpa = run_bench(lambda: torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=False))
    t_flash = run_bench(lambda: flash_attention_triton(q, k, v, causal=False))
    results.append((f"B{B} H{H} S{S} D{D}", t_sdpa, t_flash))
    print(f"{B}x{H}x{S}x{D}: SDPA {t_sdpa:.4f} ms, Flash {t_flash:.4f} ms ({t_sdpa/t_flash:.2f}x)")

## Visualize performance

In [ ]:
import matplotlib.pyplot as plt

labels = [r[0] for r in results]
sdpa_t = [r[1] for r in results]
flash_t = [r[2] for r in results]
x = range(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - w/2 for i in x], sdpa_t, w, label="PyTorch SDPA", color="#2ecc71")
ax.bar([i + w/2 for i in x], flash_t, w, label="Triton FlashAttention", color="#9b59b6")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.set_ylabel("Latency (ms)")
ax.set_title("Attention: PyTorch SDPA vs Triton FlashAttention")
ax.legend()
plt.tight_layout()
plt.show()